# Buccino et al. (2024) on dandiset 000294

Paper. Buccino AP, Damart T, Bartram J, Mandge D, Xue X, Zbili M, Gänswein T, Jaquier A,
Emmenegger V, Markram H, Hierlemann A, Van Geit W (2024). A multimodal fitting approach to construct
single-neuron models with patch-clamp and high-density microelectrode arrays. *Neural Computation*
36(7):1286-1331. [doi:10.1162/neco_a_01672](https://doi.org/10.1162/neco_a_01672). The claims file
(`claims/000294/10.1162-neco_a_01672.yaml`) was written from the version of record (CC BY 4.0), main
text, Methods, Tables 1 to 5 and the appendix; the paper has no separate supplement.

Dandiset. [DANDI:000294](https://dandiarchive.org/dandiset/000294/0.260528.0854), version
0.260528.0854 (doi:10.48324/dandi.000294/0.260528.0854). The paper cites the dandiset without a
version. This is the only published version, and the most recent; the draft has the same two files
(same sizes and blobs), so pinning it loses nothing.

Outside inputs. The paper's code repository,
[alejoe91/multimodalfitting](https://github.com/alejoe91/multimodalfitting) at commit
`24e8e6168b2d5be45c551bcc08b62fde187d01c7`, which the paper names for its scripts, model files,
morphologies and extracted features. It supplies the ground-truth model and its template (claims on
data elsewhere) and the reconstructed morphologies. It is fetched by `gtmodel.py`; nothing from it
enters the analyses of the dandiset's recordings except method parameters, cited where used.

Claims.

| Claim | Data | Tested here |
|---|---|---|
| recordings-two-cells-runs | dandiset | tested |
| ecode-protocol-structure | dandiset | tested |
| holding-potential | dandiset | tested |
| extracellular-spikes-above-50uv | dandiset | tested |
| mea-electrodes-and-rates | dandiset | tested |
| culture-age | dandiset | blocked: the files and the archive give no culture age, plating date or DIV |
| fig6-ais-from-dendrite | elsewhere (repository morphologies) | tested |
| gt-rheobase | elsewhere (repository model) | tested |
| fig3b-ap-initiates-distal-ais | elsewhere (repository model) | tested |
| fig3a-largest-channel-at-ais | elsewhere (repository template) | tested |
| table1-feature-counts | elsewhere (repository feature files) | tested |
| fig4a-strategy-balance, gt-intracellular-scores, fig4c-hyperdepol-only-single, fig4b-intra-no-difference, fig4b-extra-better-than-soma, fig4d-templates-better-than-soma, fig5-ap-distribution | elsewhere | blocked: too large (10 seeds of 600 generations per strategy; estimate below) |
| fig78a-soma-intra-all-extra, experimental-intracellular-scores | with inputs | blocked: too large (estimate below) |
| fig78b-validation-spiking, fig78c-cell2-worse-extracellular, fig10-cell2-soma-underestimated | with inputs | blocked: depend on fig78a-soma-intra-all-extra |

In [ ]:
import hashlib
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import multimodal as mm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = mm.asset_urls()
meta = {s: mm.metadata(u) for s, u in urls.items()}
runs = {s: mm.runs_in_file(u) for s, u in urls.items()}
patch = mm.load_patch(urls, runs)
print(f"dandiset {mm.DANDISET_ID} version {mm.DANDISET_VERSION}: {len(urls)} files; "
      f"patch data of {sum(len(v) for v in patch.values())} sweeps loaded in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
for s, m in meta.items():
    print(f"sub-{s} ({mm.CELLS[s]}): identifier {m['identifier']}, subject_id {m['subject_id']}, "
          f"species {m['species']}, '{m['subject_description']}'")
    print(f"   session start {m['session_start_time']}; '{m['session_description']}'")
    print(f"   subject fields in the file: {m['subject_fields']}")
    print(f"   runs in the series names: {runs[s]}; MEA series descriptions: "
          f"{[v['description'] for v in m['mea'].values()]}")
    print(f"   confocal maximum projection {m['confocal_shape']}")
print("\nThe paper's code names cell 1 cell1_211006_3148 and cell 2 cell1_211011_3436 (claims file).")

In [ ]:
import json
import urllib.request
from dandi.dandiapi import DandiAPIClient
with DandiAPIClient() as client:
    raw = client.get_dandiset(mm.DANDISET_ID, mm.DANDISET_VERSION).get_raw_metadata()
print("archive summary:", {k: raw["assetsSummary"].get(k) for k in ("numberOfFiles", "numberOfSubjects", "variableMeasured")})
for r in raw.get("relatedResource", []):
    print("related resource:", r.get("relation"), r.get("url"), "|", r.get("name"))
    if "doi.org/" in r.get("url", ""):
        doi = r["url"].split("doi.org/")[1]
        title = json.load(urllib.request.urlopen(f"https://api.crossref.org/works/{doi}", timeout=60))["message"]["title"][0]
        print(f"   Crossref title of {doi}: {title}")
print("paper: doi:10.1162/neco_a_01672")

The archive's metadata name the paper by title but link it to doi:10.1162/neco_a_01676, which
Crossref gives as another article; the paper is doi:10.1162/neco_a_01672.

Each file holds one neuron; the identifiers are the code's cell names (cell 1 is sub-3148, cell 2 is
sub-3436). Cell 2's runs are named run1 to run4, while the descriptions of its MEA series say run3 to
run6: the export renamed the original runs 3 to 6, as the code's `run_map` does. The subject has no
age, date of birth or culture date, so the days in vitro cannot be read from the files.

In [ ]:
rows = []
for s, sweeps in patch.items():
    for d in sweeps:
        dt = np.diff(d["t"])
        rows.append(dict(cell=mm.CELLS[s], run=d["run"], protocol=d["protocol"], sweep=d["sweep"], n=len(d["t"]),
                         dt_us=round(float(np.median(dt)) * 1e6, 3), dt_spread_us=float(np.ptp(dt)) * 1e6,
                         same_t=d["same_t"], v_min=float(d["v"].min()), v_max=float(d["v"].max()),
                         i_min=float(d["i"].min()), i_max=float(d["i"].max()),
                         t0=float(d["t"][0]), t1=float(d["t"][-1])))
sw = pd.DataFrame(rows)
print("Sweeps per cell, run and protocol:")
print(sw.pivot_table(index=["cell", "run"], columns="protocol", values="sweep", aggfunc="count").to_string())
print("\nSampling interval of the patch timestamps (us) by protocol (median; spread within a sweep):")
print(sw.groupby(["cell", "protocol"]).agg(dt_us=("dt_us", "median"), dt_min=("dt_us", "min"), dt_max=("dt_us", "max"),
                                          spread_max=("dt_spread_us", "max")).to_string())
print("\nStimulus and response share their timestamps in every sweep:", bool(sw.same_t.all()))
print("\nRanges by cell:")
print(sw.groupby("cell").agg(v_min=("v_min", "min"), v_max=("v_max", "max"), i_min=("i_min", "min"),
                             i_max=("i_max", "max")).round(1).to_string())
for s, m in meta.items():
    print(f"{mm.CELLS[s]}: response unit '{m['patch_attrs']['data_unit']}', conversion {m['patch_attrs']['data_conversion']}, "
          f"gain {m['patch_attrs']['gain']}; stimulus unit '{m['stim_attrs']['data_unit']}', conversion {m['stim_attrs']['data_conversion']}")

The responses are stated in volts with conversion 1, but range from about -100 to +46: they are in
millivolts. The stimuli are stated in amperes, but the holding levels and steps (tens to hundreds)
are picoamperes: the IDthres steps of cell 1 start at 60 above holding, which is 50% of the rheobase
scale of a cortical neuron in pA, not in nA or A. The analyses use mV and pA. The timestamps are
regular within every sweep. Their interval is 50 us for IDthres (20 kHz), 49 us for the other
protocols (20.41 kHz) and 8 us for APWaveform (125 kHz). The stimulus step durations measured on these
timestamps are the nominal ones (3600 ms, 1350 ms, 270 ms, below), so the timestamps are consistent
with the protocols, and the intervals are taken as the real sampling intervals.

In [ ]:
for s, m in meta.items():
    e = m["electrodes"]
    print(f"{mm.CELLS[s]}: electrodes table {len(e['id'])} rows; columns {sorted(e)}")
    print(f"   x, y, z all NaN: {np.isnan(e['x']).all()}, {np.isnan(e['y']).all()}, {np.isnan(e['z']).all()}; "
          f"rel_x {e['rel_x'].min()} to {e['rel_x'].max()} um, rel_y {e['rel_y'].min()} to {e['rel_y'].max()} um")
    pitch = np.unique(np.round(np.diff(np.unique(e["rel_x"])), 2))
    print(f"   smallest spacing of rel_x values {pitch.min()} um; gain_to_uV {np.unique(e['gain_to_uV'])}; "
          f"electrode numbers {int(e['electrode'].min())} to {int(e['electrode'].max())} of 26,400")
    for name, v in m["mea"].items():
        print(f"   {name}: {v['shape']} {v['dtype']} chunks {v['chunks']}, conversion {v['conversion']:.4g} {v['unit']}, "
              f"{v['n_electrodes']} electrodes, t {v['t0']:.2f} to {v['t1']:.2f} s, rate {v['rate']:.2f} Hz")

The MEA series are 10-bit counts (values up to 1023) with a conversion of 3.147 uV per count, the
`gain_to_uV` of the electrodes table, at 20 kHz. Electrode positions are in `rel_x` and `rel_y` on a
17.5 um grid (Methods 5.6.2); `x`, `y` and `z` are empty.

In [ ]:
# Patch and MEA of a run share one clock: patch sweeps fall inside the run's MEA series.
for s, m in meta.items():
    for r in runs[s]:
        sub = sw[(sw.cell == mm.CELLS[s]) & (sw.run == r)]
        mea = m["mea"][f"ElectricalSeries_{r}"]
        print(f"{mm.CELLS[s]} {r}: patch {sub.t0.min():.2f} to {sub.t1.max():.2f} s; MEA {mea['t0']:.2f} to {mea['t1']:.2f} s")

# The icephys grouping tables
for s, m in meta.items():
    idx = np.r_[0, m["seq_index"]]
    members = set(m["seq_members"].tolist())
    missing = sorted(set(range(len(m["ic_protocol_name"]))) - members)
    miss_prot = Counter(m["ic_protocol_name"][missing])
    print(f"\n{mm.CELLS[s]}: {len(m['ic_protocol_name'])} intracellular recordings; {len(m['seq_type'])} sequential recordings "
          f"(stimulus types {dict(Counter(m['seq_type']))}); {m['n_repetitions']} repetitions; "
          f"{len(missing)} recordings in no sequential recording, by protocol {dict(miss_prot)}")
    print("   table columns protocol_name and run agree with the series names:",
          bool((sw[sw.cell == mm.CELLS[s]].sort_values("sweep").protocol.values == m["ic_protocol_name"]).all()))

# Recordings deposited twice: checksums of every patch sweep across both files
hashes = Counter()
for s, sweeps in patch.items():
    for d in sweeps:
        hashes[hashlib.sha1(d["v"].tobytes()).hexdigest()] += 1
print(f"\nPatch sweeps: {sum(hashes.values())}, distinct voltage traces: {len(hashes)}")

Within each run the patch sweeps fall inside the MEA series, on one clock that starts at the first
run. The grouping tables are incomplete: in each run the first sweep of every protocol after the
first, and the PosCheops sweep, belong to no sequential recording (the export loop of
`0-export-to-nwb.ipynb` starts a new group without its first sweep). The `protocol_name` and `run`
columns are complete and agree with the series names, so the analyses use these. No voltage trace
is deposited twice; the MEA runs are compared below by checksums of their raw segments.

In [ ]:
# Denoising of every run (code's remove_blank_channels), with checksums of the raw segments.
t_mea = time.time()
blank, mea_hash = {}, {}
for s, m in meta.items():
    for r in runs[s]:
        name = f"ElectricalSeries_{r}"
        blank[(s, r)], mea_hash[(s, r)] = mm.blank_channels(urls[s], name, m["mea"][name]["shape"][0])
dn = pd.DataFrame([dict(cell=mm.CELLS[s], run=r, channels=len(b), blank=int(b.sum()), kept=int((~b).sum()))
                   for (s, r), b in blank.items()])
print(dn.to_string(index=False))
allh = [h for v in mea_hash.values() for h in v]
print(f"\nRaw MEA segments compared: {len(allh)}, distinct: {len(set(allh))}  ({time.time() - t_mea:.0f} s)")

Every run of cell 1 has 678 channels and cell 2 675, but many are constant (blank) in some second of
the run; these are the channels the code removes before computing templates. No raw MEA segment is
repeated between runs or files.

In [ ]:
# Clock check: spike-triggered MEA waveforms at the patch spike times, and at times shifted by 100 ms.
def run_spikes(s, r, include_poscheops=False):
    out = []
    for d in patch[s]:
        if d["run"] == r and (include_poscheops or d["protocol"] != "poscheops"):
            out.append(mm.peak_times(d["t"], d["v"]))
    return np.concatenate(out)

mea_t = {}
def mea_timestamps(s, r):
    if (s, r) not in mea_t:
        mea_t[(s, r)] = mm.open_file(urls[s])["acquisition"][f"ElectricalSeries_{r}"]["timestamps"][:]
    return mea_t[(s, r)]

clock_rows = []
for s in urls:
    r = runs[s][0]
    name = f"ElectricalSeries_{r}"
    info = meta[s]["mea"][name]
    sp = run_spikes(s, r)
    for shift in (0.0, 0.1, -0.1):
        wf, used, wide = mm.mea_waveforms(urls[s], name, mea_timestamps(s, r), sp + shift, info["conversion"] * 1e6,
                                          info["chunks"][0])
        T = mm.template_from_waveforms(wf, wide, keep=~blank[(s, r)])
        a = mm.amplitudes(T["median"], T["max_channel"])
        clock_rows.append(dict(cell=mm.CELLS[s], run=r, shift_ms=shift * 1e3, spikes=len(used),
                               max_channel_neg_peak_uV=round(a["neg_peak"], 1), max_channel_ptp_uV=round(a["ptp"], 1)))
print(pd.DataFrame(clock_rows).to_string(index=False))

With the patch spike times, the template has a spike of 189.2 uV (cell 1) and 159.2 uV (cell 2) on
its largest channel; with the same times shifted by 100 ms in either direction, the largest value
over more than 500 channels is 19.5 to 23.5 uV, the noise left after averaging. So the
patch and MEA series of a run share their clock, at least to well within the 2 ms search window of
the centring. This is shown for the first run of each cell; the other runs are checked the same way
by their templates below.

## recordings-two-cells-runs

> The experimental data are simultaneous patch-clamp and HD-MEA recordings of two cultured neurons,
> each stimulated with the eCode protocols repeated in four to six runs of about 200 s.

Criterion: two neurons, each with 4 to 6 runs, and the median run duration (first to last timestamp
of a run's patch series) within 200 +/- 20 s.

In [ ]:
dur = sw.groupby(["cell", "run"]).agg(first=("t0", "min"), last=("t1", "max"))
dur["duration_s"] = dur["last"] - dur["first"]
dur["mea_s"] = [meta[s]["mea"][f"ElectricalSeries_{r}"]["t1"] - meta[s]["mea"][f"ElectricalSeries_{r}"]["t0"]
                for s in urls for r in runs[s]]
print(dur.round(2).to_string())
n_runs = dur.groupby("cell").size()
med = float(dur.duration_s.median())
print(f"\nneurons (NWB files): {len(urls)}; runs per neuron: {n_runs.to_dict()}")
print(f"median run duration (patch series): {med:.1f} s (range {dur.duration_s.min():.1f} to {dur.duration_s.max():.1f}); "
      f"MEA series {dur.mea_s.min():.1f} to {dur.mea_s.max():.1f} s")
print("runs per cell within 4 to 6:", bool(((n_runs >= 4) & (n_runs <= 6)).all()), "; duration within 180 to 220 s:", 180 <= med <= 220)

**Result: reproduced.** The dandiset holds two neurons, cell 1 with 5 runs and cell 2 with 4 (the
runs the paper's code used: runs 1 to 5 of cell 1 and 3 to 6 of cell 2, renamed). The median run
lasts 195.8 s from the first to the last patch sample (195.7 to 195.8 s, the same as the MEA series),
within 200 +/- 20 s.

## ecode-protocol-structure

> Each run applies the eCode protocols with the sweep numbers, step durations and amplitudes (in
> percent of the estimated rheobase) that the Methods list.

Criterion: in every run, all eight protocols with the Methods' sweep counts and durations (within
10%) and every step amplitude within 10% of its stated percentage of the IDthres scale (sweep k of
IDthres at 50 + 4k %); IV under reading (a) 11 steps from -140 to 60% or (b) 9 steps from -140 to
20%. Amplitudes are measured above the holding level. For a stated 0% (IV, reading a) a relative
tolerance is not defined; 10% of the IV increment (2% of the scale) is used (deviation).

In [ ]:
EXPECTED = {   # Methods 5.2: (sweeps, [(duration ms, % per sweep), ...] per phase)
    "idthres": (21, [(270, [50 + 4 * k for k in range(21)])]),
    "firepattern": (2, [(3600, [120, 200])]),
    "iv_a": (11, [(3000, list(range(-140, 61, 20)))]),
    "iv_b": (9, [(3000, list(range(-140, 21, 20)))]),
    "idrest": (11, [(1350, list(range(50, 301, 25)))]),
    "apwaveform": (6, [(50, list(range(200, 351, 30)))]),
    "hyperdepol": (4, [(450, [-40, -80, -120, -160]), (270, [100] * 4)]),
    "sahp": (4, [(250, [40] * 4), (225, [150, 200, 250, 300]), (450, [40] * 4)]),
}

def scale_of(s, r):
    ids = [d for d in patch[s] if d["run"] == r and d["protocol"] == "idthres"]
    amps = [mm.segments(d)[1][2] for d in ids]
    return float(np.median([a / ((50 + 4 * k) / 100) for k, a in enumerate(amps)])), amps

def ok_amp(meas, exp):
    return abs(meas - exp) <= (0.1 * abs(exp) if exp != 0 else 2.0)

rows, detail = [], []
for s in urls:
    for r in runs[s]:
        scale, _ = scale_of(s, r)
        by = {}
        for d in patch[s]:
            if d["run"] == r:
                by.setdefault(d["protocol"], []).append(d)
        for prot, key_list in [("idthres", ["idthres"]), ("firepattern", ["firepattern"]), ("iv", ["iv_a", "iv_b"]),
                               ("idrest", ["idrest"]), ("apwaveform", ["apwaveform"]), ("hyperdepol", ["hyperdepol"]),
                               ("sahp", ["sahp"])]:
            sweeps = sorted(by.get(prot, []), key=lambda d: d["sweep"])
            phases = [[seg for seg in mm.segments(d) if abs(seg[2]) > 0.02 * scale] for d in sweeps]
            for key in key_list:
                n_exp, exp_phases = EXPECTED[key]
                count_ok = len(sweeps) == n_exp
                dur_ok = amp_ok = count_ok
                if count_ok:
                    for k, ph in enumerate(phases):
                        if len(ph) != len(exp_phases):
                            # a 0% step has no phase above holding
                            if not (len(ph) == 0 and all(e[1][k] == 0 for e in exp_phases)):
                                dur_ok = amp_ok = False
                            continue
                        for (start, dms, lev), (edur, eperc) in zip(ph, exp_phases):
                            dur_ok &= abs(dms - edur) <= 0.1 * edur
                            amp_ok &= ok_amp(100 * lev / scale, eperc[k])
                            detail.append(dict(cell=mm.CELLS[s], run=r, protocol=key, sweep=k, duration_ms=round(dms, 1),
                                               expected_ms=edur, percent=round(100 * lev / scale, 1), expected_percent=eperc[k]))
                rows.append(dict(cell=mm.CELLS[s], run=r, protocol=key, scale_pA=round(scale, 1), sweeps=len(sweeps),
                                 expected_sweeps=n_exp, count_ok=count_ok, durations_ok=bool(dur_ok), amplitudes_ok=bool(amp_ok)))
        # PosCheops: three ramps to 300% with rise and fall of 4, 2, 1.33 s
        pc = by.get("poscheops", [])
        rp = mm.ramps(pc[0]) if len(pc) == 1 else []
        exp_r = [4000, 2000, 1333]
        r_ok = len(rp) == 3 and all(abs(a[1] - e) <= 0.1 * e and abs(a[2] - e) <= 0.1 * e for a, e in zip(rp, exp_r))
        a_ok = len(rp) == 3 and all(ok_amp(100 * a[3] / scale, 300) for a in rp)
        rows.append(dict(cell=mm.CELLS[s], run=r, protocol="poscheops", scale_pA=round(scale, 1), sweeps=len(pc),
                         expected_sweeps=1, count_ok=len(pc) == 1, durations_ok=bool(r_ok), amplitudes_ok=bool(a_ok)))
        for k, a in enumerate(rp):
            detail.append(dict(cell=mm.CELLS[s], run=r, protocol="poscheops", sweep=k, duration_ms=round(a[1], 1),
                               expected_ms=exp_r[k], percent=round(100 * a[3] / scale, 1), expected_percent=300,
                               fall_ms=round(a[2], 1)))
ec = pd.DataFrame(rows)
det = pd.DataFrame(detail)
print("IDthres scale (100%) per run, pA:")
print(ec.drop_duplicates(["cell", "run"])[["cell", "run", "scale_pA"]].to_string(index=False))
print("\nPer protocol, every run:")
print(ec.groupby(["cell", "protocol"])[["count_ok", "durations_ok", "amplitudes_ok"]].all().to_string())
print("\nMeasured durations and percentages, range over runs and sweeps:")
print(det.groupby(["protocol", "expected_ms"]).agg(dur_min=("duration_ms", "min"), dur_max=("duration_ms", "max")).to_string())
det["err_percent"] = det.percent - det.expected_percent
print("\nLargest deviation from the stated percentage, by protocol (percentage points):")
print(det.groupby(["cell", "protocol"]).err_percent.agg(lambda x: x.abs().max()).round(2).to_string())

In [ ]:
ok_a = ec[ec.protocol != "iv_b"][["count_ok", "durations_ok", "amplitudes_ok"]].all()
ok_b = ec[ec.protocol != "iv_a"][["count_ok", "durations_ok", "amplitudes_ok"]].all()
print("reading (a), IV -140 to 60%:", ok_a.to_dict())
print("reading (b), IV -140 to 20%:", ok_b.to_dict())
iv_pc = det[det.protocol == "iv_a"].groupby("sweep").percent.median()
print("IV, median percent per sweep over runs and cells:", iv_pc.round(1).tolist())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s in urls:
    d = [x for x in patch[s] if x["run"] == runs[s][0] and x["protocol"] == "sahp"][-1]
    ax[0].plot((d["t"] - d["t"][0]) * 1e3, d["i"] - mm.holding_current(d), label=mm.CELLS[s])
    d = [x for x in patch[s] if x["run"] == runs[s][0] and x["protocol"] == "poscheops"][0]
    ax[1].plot((d["t"] - d["t"][0]), d["i"] - mm.holding_current(d), label=mm.CELLS[s])
ax[0].set(title="sAHP 300% sweep, first run", xlabel="ms", ylabel="pA above holding")
ax[1].set(title="PosCheops, first run", xlabel="s")
ax[0].legend(); plt.tight_layout(); plt.show()
print("All plotted values are inside the axes (automatic limits).")

**Result: reproduced.** In every run of both cells all eight protocols are present with the Methods'
sweep counts (21, 2, 11, 11, 6, 4, 4 and 1), step and ramp durations within 10% and amplitudes within
10% of their stated percentages of the IDthres scale (120.4 to 120.8 pA for cell 1, 301.2 to
301.7 pA for cell 2); the largest deviation is 1.2 percentage points. IV runs from -140 to 60% in 11
steps, reading (a); reading (b), 9 steps to 20%, fails on the sweep count. The PosCheops ramps reach
300% and rise in 3925 to 3975, 1965 to 1988 and 1307 to 1322 ms (measured from 5 pA above holding),
within 10% of 4, 2 and 1.33 s.

## holding-potential

> The holding current was set so that the read-out membrane potential was about -70 mV, which is
> about -84 mV after liquid junction potential correction.

Criterion: per sweep, the median membrane potential before the stimulus onset; median over sweeps
per cell; reading (a) uncorrected, -70 +/- 7 mV; (b) corrected, -84 +/- 8.4 mV; reproduced when both
cells match under either reading. The onset is taken from the stimulus itself (the first change of
the injected current). The code's `ton` (400 ms for IDrest) is 50 ms after the onset in these files,
so it is shown as a variant, though its window includes 50 ms of the step.

In [ ]:
hp = []
for s in urls:
    for d in patch[s]:
        on = mm.onset_ms(d)
        tt = (d["t"] - d["t"][0]) * 1e3
        pre = d["v"][tt < on]
        pre_code = d["v"][tt < mm.CODE_TON_MS[d["protocol"]]]
        hp.append(dict(cell=mm.CELLS[s], run=d["run"], protocol=d["protocol"], onset_ms=round(on, 1),
                       v_pre=float(np.median(pre)), v_pre_code_ton=float(np.median(pre_code)),
                       i_hold=mm.holding_current(d)))
hp = pd.DataFrame(hp)
print("Stimulus onset by protocol (ms from sweep start), code's ton:")
print(hp.groupby("protocol").onset_ms.agg(["min", "max"]).assign(code_ton=lambda x: x.index.map(mm.CODE_TON_MS)).to_string())
summ = hp.groupby("cell").agg(v_pre_median=("v_pre", "median"), v_pre_min=("v_pre", "min"), v_pre_max=("v_pre", "max"),
                              v_code_ton=("v_pre_code_ton", "median"), i_hold_pA=("i_hold", "median"), sweeps=("v_pre", "size"))
print("\nPer cell (mV, pA):")
print(summ.round(2).to_string())
print("\nBy run:")
print(hp.groupby(["cell", "run"]).v_pre.median().round(2).to_string())
for cell, v in summ.v_pre_median.items():
    print(f"{cell}: {v:.2f} mV; reading (a) |v + 70| <= 7: {abs(v + 70) <= 7}; reading (b) |v + 84| <= 8.4: {abs(v + 84) <= 8.4}")

**Result: reproduced.** The median pre-stimulus potential is -69.54 mV for cell 1 and -69.12 mV for
cell 2 (median holding currents 12.24 and -36.72 pA), within -70 +/- 7 mV, reading (a); run medians
range from -71.0 to -66.6 mV. Neither cell is within -84 +/- 8.4 mV, so the stored voltages are not
corrected for the junction potential. The code's `ton` window gives the same values to within
0.1 mV.

## extracellular-spikes-above-50uv

> Only cells whose extracellular spikes were at least 50 uV were kept, so both recorded cells show
> extracellular spikes of 50 uV or more.

Criterion: the template of each cell by the template method; reading (a) the absolute negative peak
and (b) the peak-to-peak on the largest channel; reproduced when both cells reach 50 uV under either
reading. Templates are computed for every run (Methods 5.7.1), the run with the most channels after
denoising is used, and the others are shown.

In [ ]:
t_tpl = time.time()
templates, trow = {}, []
for s in urls:
    for r in runs[s]:
        name = f"ElectricalSeries_{r}"
        info = meta[s]["mea"][name]
        sp = run_spikes(s, r)
        wf, used, wide = mm.mea_waveforms(urls[s], name, mea_timestamps(s, r), sp, info["conversion"] * 1e6, info["chunks"][0])
        T = mm.template_from_waveforms(wf, wide, keep=~blank[(s, r)])
        templates[(s, r)] = T
        a, am = mm.amplitudes(T["median"], T["max_channel"]), mm.amplitudes(T["mean"], T["max_channel"])
        ptp = np.ptp(T["median"], axis=0)
        trow.append(dict(cell=mm.CELLS[s], run=r, patch_spikes=len(sp), waveforms=T["n_waveforms"], kept=T["n_kept"],
                         channels_after_denoising=len(T["channels"]), channels_ptp_ge_5uV=int((ptp >= 5).sum()),
                         max_channel_electrode=meta[s]["electrodes"]["channel_name"][T["channels"][T["max_channel"]]],
                         neg_peak_uV=round(a["neg_peak"], 1), ptp_uV=round(a["ptp"], 1),
                         neg_peak_mean_uV=round(am["neg_peak"], 1), ptp_mean_uV=round(am["ptp"], 1),
                         shift_median_ms=float(np.median(T["shifts_ms"]))))
        del wf
tp = pd.DataFrame(trow)
print(tp.to_string(index=False))
print(f"({time.time() - t_tpl:.0f} s)")
chosen = tp.loc[tp.groupby("cell").channels_after_denoising.idxmax()]
print("\nRun used per cell (most channels after denoising; first run on ties):")
print(chosen[["cell", "run", "channels_after_denoising", "neg_peak_uV", "ptp_uV", "neg_peak_mean_uV", "ptp_mean_uV"]].to_string(index=False))
for _, c in chosen.iterrows():
    print(f"{c.cell}: (a) negative peak {c.neg_peak_uV} uV >= 50: {c.neg_peak_uV >= 50}; (b) peak-to-peak {c.ptp_uV} uV >= 50: {c.ptp_uV >= 50}")
code_chan = {"3148": "ch885", "3436": "ch384"}   # the channel 0-export-to-nwb.ipynb plots for each cell
for s in urls:
    names = meta[s]["electrodes"]["channel_name"]
    k = [i for i, n in enumerate(names) if n.split()[0] == code_chan[s]][0]
    r = chosen[chosen.cell == mm.CELLS[s]].run.iloc[0]
    T = templates[(s, r)]
    if k in T["channels"]:
        j = int(np.where(T["channels"] == k)[0][0])
        a = mm.amplitudes(T["median"], j)
        print(f"{mm.CELLS[s]}: the export notebook's channel {code_chan[s]} ({names[k]}): negative peak {a['neg_peak']:.1f} uV, ptp {a['ptp']:.1f} uV")
    else:
        print(f"{mm.CELLS[s]}: the export notebook's channel {code_chan[s]} is removed as blank")
print(f"\nOver all runs: negative peak {tp.neg_peak_uV.min()} to {tp.neg_peak_uV.max()} uV; peak-to-peak {tp.ptp_uV.min()} to {tp.ptp_uV.max()} uV")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, (_, c) in zip(axs, chosen.iterrows()):
    s = [k for k, v in mm.CELLS.items() if v == c.cell][0]
    T = templates[(s, c.run)]
    tt = np.arange(T["median"].shape[0]) / 20.0 - mm.MS_BEFORE
    ch = T["max_channel"]
    ax.plot(tt, T["median"][:, ch], "k", label="median (Methods)")
    ax.plot(tt, T["mean"][:, ch], "C1--", label="mean (code)")
    ax.axhline(-50, color="gray", lw=0.8)
    ax.set(title=f"{c.cell} {c.run}, largest channel", xlabel="ms from extracellular trough", ylabel="uV")
    ax.legend()
    print(f"{c.cell}: plotted range {T['median'][:, ch].min():.1f} to {T['median'][:, ch].max():.1f} uV, all inside the axes")
plt.tight_layout(); plt.show()

**Result: reproduced.** On the run with the most channels after denoising, the largest channel's
template reaches a negative peak of 217.0 uV (peak-to-peak 274.0 uV) for cell 1, run 4, and
159.2 uV (192.2 uV) for cell 2, run 1, at least 50 uV under both readings. Every run of both cells is
above 50 uV under both readings (negative peaks 159.2 to 218.0 uV), and the code's mean template is
higher by up to 13.3 uV. The channel the paper's export notebook plots for cell 1 (ch885) is the
largest channel found here; for cell 2 it plots ch384, an adjacent electrode (156.8 uV).

## mea-electrodes-and-rates

> The HD-MEA recorded more than 500 electrodes near the pipette simultaneously, out of at most 1024,
> at 20 kHz, and the patch signal was digitized at 20 kHz.

Criterion: per cell, more than 500 and at most 1024 recorded electrodes, an MEA rate of 20 kHz, and a
patch rate of 20 kHz for every protocol (APWaveform at 20 or 200 kHz); expected rates with tolerance 0.

In [ ]:
er = []
for s, m in meta.items():
    for name, v in m["mea"].items():
        er.append(dict(cell=mm.CELLS[s], series=name, electrodes=v["n_electrodes"], mea_rate_Hz=round(v["rate"], 3)))
er = pd.DataFrame(er)
print(er.to_string(index=False))
pr = sw.groupby(["cell", "protocol"]).dt_us.median().rename("dt_us").reset_index()
pr["patch_rate_Hz"] = (1e6 / pr.dt_us).round(1)
print("\nPatch sampling rate from the timestamps:")
print(pr.pivot(index="protocol", columns="cell", values="patch_rate_Hz").to_string())
el_ok = bool(((er.electrodes > 500) & (er.electrodes <= 1024)).all())
mea_ok = bool((er.mea_rate_Hz.round() == 20000).all())
non_ap = pr[pr.protocol != "apwaveform"]
patch_ok = bool((non_ap.patch_rate_Hz.round() == 20000).all())
ap_ok = bool(pr[pr.protocol == "apwaveform"].patch_rate_Hz.round().isin([20000, 200000]).all())
print(f"\nelectrodes in (500, 1024]: {el_ok}; MEA 20 kHz: {mea_ok}; patch 20 kHz for every protocol but APWaveform: {patch_ok}; "
      f"APWaveform at 20 or 200 kHz: {ap_ok}")
print("Protocols at 20 kHz:", sorted(non_ap[non_ap.patch_rate_Hz.round() == 20000].protocol.unique()))

**Result: partially reproduced.** Each cell's MEA recorded 678 (cell 1) and 675 (cell 2) electrodes at
20 kHz, as stated. The patch timestamps give 20 kHz only for IDthres; the other protocols are at
20.41 kHz (49 us intervals) and APWaveform at 125 kHz (8 us), neither 20 nor 200 kHz. The intervals
agree with the nominal stimulus durations, so they are the sampling of the files as deposited.

## culture-age

> The cultured cells were recorded between days in vitro 18 and 24.

Not tested: the subject has only an id, a species and a description ("Rat cortical embryonic cell
culture"); neither the files nor the archive's metadata give an age, a plating date or days in vitro
(fields printed in the data check). The session dates (2021-10-06 and 2021-10-11) are 5 days apart,
which fits either range of the criterion but does not give the age.

## fig6-ais-from-dendrite

> In both cells the axon initial segment does not leave the soma but a dendrite (an axon-bearing
> dendrite).

Data elsewhere: the reconstructed morphologies `cell_models/<cell>/morphology_corrected.swc` of the
repository at the pinned commit. Criterion: the AIS is the path labelled "apic" (type 4, Methods
5.7.2); reproduced when, in both cells, the first AIS point's parent is a dendrite point (type 3),
not the soma (type 1).

In [ ]:
import gtmodel as gm
root = gm.fetch_repo()
for s, code_name in mm.CODE_NAMES.items():
    a = gm.read_swc(root / "cell_models" / code_name / "morphology_corrected.swc")
    ids, typ, par = a[:, 0].astype(int), a[:, 1].astype(int), a[:, 6].astype(int)
    pos = {i: k for k, i in enumerate(ids)}
    counts = dict(Counter(typ.tolist()))
    roots = [k for k in np.where(typ == 4)[0] if par[k] == -1 or typ[pos[par[k]]] != 4]
    print(f"{mm.CELLS[s]} ({code_name}): points by type {counts}")
    for k in roots:
        p = pos[par[k]]
        soma = np.where(typ == 1)[0][0]
        print(f"   AIS starts at point {ids[k]}; its parent {ids[p]} has type {typ[p]}; "
              f"distance of the AIS start from the soma point {np.linalg.norm(a[k, 2:5] - a[soma, 2:5]):.1f} um")
    ax_roots = [k for k in np.where(typ == 2)[0] if typ[pos[par[k]]] != 2]
    print(f"   axon (type 2) starts from a point of type {[int(typ[pos[par[k]]]) for k in ax_roots]}")

**Result: reproduced.** In both reconstructions the single AIS path (type 4) starts from a dendrite
point (type 3), not from the soma, and the axon (type 2) starts from the end of the AIS.

## gt-rheobase

> The ground-truth model's rheobase, estimated with a 270 ms step, is 150 pA.

Data elsewhere: the released hay_ais model of the repository. Criterion: the lowest amplitude of
270 ms steps that elicits a spike, with the step series of the ground-truth notebook (from 0.15 nA by
0.005 nA). Because that series starts at the stated value, a series from 0.10 nA is also run.

In [ ]:
t_sim = time.time()
rheo_nb = gm.rheobase()
rheo_low = gm.rheobase(step_min=0.10)
print("notebook series (nA, spikes):", rheo_nb)
print("series from 0.10 nA (nA, spikes):", rheo_low)
print(f"rheobase: notebook series {rheo_nb[-1][0] * 1000:.0f} pA; lower series {rheo_low[-1][0] * 1000:.0f} pA; "
      f"within 150 +/- 15 pA: {abs(rheo_nb[-1][0] * 1000 - 150) <= 15}, {abs(rheo_low[-1][0] * 1000 - 150) <= 15} "
      f"({time.time() - t_sim:.0f} s)")
p300 = gm.protocol_json()["IDrest_300"]["stimuli"][1]
print(f"IDrest 300% in the repository's protocol file: {p300['amp']:.3f} nA, {p300['thresh_perc']:.0f}% -> 100% = {p300['amp'] / 3 * 1000:.0f} pA")

**Result: reproduced.** With the ground-truth notebook's series the first step, 150 pA, fires one
spike. Starting lower, the model first fires at 140 pA, also within 150 +/- 15 pA. The repository's
IDrest 300% step of 0.45 nA implies the same 150 pA scale.

## fig3b-ap-initiates-distal-ais

> In the ground-truth model the action potential starts in the distal AIS and travels back toward the
> soma.

Criterion: IDrest 300% on the released model, membrane potential at the soma, the middle and the end
of the AIS, cut-outs averaged around the somatic peaks; readings (a) the time the averaged AP crosses
-20 mV and (b) the time of its peak; reproduced when under both the distal AIS comes first, then the
middle, then the soma.

In [ ]:
resp = gm.run_step(p300["amp"], p300["delay"], p300["duration"], p300["totduration"], ais=True)
grid, cut, n_cut = gm.averaged_cutouts(resp)
fine = np.arange(grid[0], grid[-1], 0.0005)
order = []
for name in ("ais_end_v.v", "ais_mid_v.v", "soma.v"):
    v = np.interp(fine, grid, cut[name])
    k = np.where(v >= -20)[0][0]
    order.append(dict(location=name.split(".")[0], cross_minus20_ms=round(float(fine[k]), 4),
                      peak_ms=round(float(fine[np.argmax(v)]), 4), peak_mV=round(float(v.max()), 1)))
order = pd.DataFrame(order)
print(f"spikes averaged: {n_cut} (somatic peaks of IDrest 300%, first and last left out); times relative to the somatic peak")
print(order.to_string(index=False))
a_ok = order.cross_minus20_ms.is_monotonic_increasing and order.cross_minus20_ms.is_unique
b_ok = order.peak_ms.is_monotonic_increasing and order.peak_ms.is_unique
print(f"distal AIS, then middle AIS, then soma: reading (a) {a_ok}; reading (b) {b_ok}")
print(f"distal minus middle: crossing {order.cross_minus20_ms[0] - order.cross_minus20_ms[1]:.4f} ms, peak {order.peak_ms[0] - order.peak_ms[1]:.4f} ms")
fig, ax = plt.subplots(figsize=(6, 3.5))
for name in cut:
    ax.plot(grid, cut[name], label=name.split(".")[0])
ax.set(xlim=(-1, 2), xlabel="ms from somatic peak", ylabel="mV"); ax.legend(); plt.show()
print(f"plotted window -1 to 2 ms of the -3 to 10 ms cut-out; values {min(v.min() for v in cut.values()):.1f} to {max(v.max() for v in cut.values()):.1f} mV")

**Result: reproduced.** Averaged over 12 spikes, the AP crosses -20 mV first at the end of the AIS
(-0.360 ms from the somatic peak), then at its middle (-0.355 ms) and then at the soma (-0.185 ms),
and peaks in the same order (-0.24, -0.22 and 0.01 ms). The distal lead over the middle of the AIS is
small (0.005 ms at the crossing, 0.02 ms at the peak), but both readings give the order of the claim.

## fig3a-largest-channel-at-ais

> On the simulated 80-electrode MEA, the largest extracellular signal is on the electrode closest to
> the AIS.

Data elsewhere: the ground-truth template `cell_models/hay_ais/fitting/extracellular/template.npy`
with its `locations.npy` (20 by 4 electrodes, 50 um pitch). Criterion: the channel with the largest
peak-to-peak amplitude is the electrode nearest the AIS midpoint in the plane; the largest absolute
negative peak is shown too. The AIS midpoint is taken from the released model's morphology, relative
to the soma centre (LFPy places the soma midpoint at the origin).

In [ ]:
ext = root / "cell_models" / "hay_ais" / "fitting" / "extracellular"
eap, loc, fs_gt = np.load(ext / "template.npy"), np.load(ext / "locations.npy"), float(np.load(ext / "fs.npy"))
print(f"template {eap.shape} (channels, samples) at {fs_gt:.0f} Hz; electrodes x {loc[:, 0].min()} to {loc[:, 0].max()}, "
      f"y {loc[:, 1].min()} to {loc[:, 1].max()} um, z {np.unique(loc[:, 2])}")
pts = gm.section_points()
soma = [v for k, v in pts.items() if k.endswith("soma[0]")][0]
ais = [v for k, v in pts.items() if k.endswith("ais[0]")][0]
centre = soma.mean(axis=0)
ais_mid = ais[len(ais) // 2] - centre
ais_ends = ais[[0, -1]] - centre
print(f"AIS from {ais_ends[0, :2].round(1)} to {ais_ends[1, :2].round(1)} um (x, y) relative to the soma centre; midpoint {ais_mid[:2].round(1)}")
d_ais = np.linalg.norm(loc[:, :2] - ais_mid[:2], axis=1)
ptp = np.ptp(eap, axis=1)
neg = -eap.min(axis=1)
near = np.argsort(d_ais)[:3]
print("electrodes nearest the AIS midpoint:", [(int(k), loc[k, :2].tolist(), round(float(d_ais[k]), 1)) for k in near])
print(f"largest peak-to-peak: channel {int(np.argmax(ptp))} at {loc[np.argmax(ptp), :2].tolist()} ({ptp.max():.4g}); "
      f"largest negative peak: channel {int(np.argmax(neg))} at {loc[np.argmax(neg), :2].tolist()} ({neg.max():.4g}), "
      f"in the template's units")
print("rank of the nearest electrode by peak-to-peak:", int(np.where(np.argsort(-ptp) == near[0])[0][0]) + 1,
      "; ratio of the second-largest to the largest peak-to-peak:", round(float(np.sort(ptp)[-2] / ptp.max()), 3))
print("largest peak-to-peak on the nearest electrode:", int(np.argmax(ptp)) == int(near[0]),
      "; largest negative peak on the nearest electrode:", int(np.argmax(neg)) == int(near[0]))

**Result: reproduced.** The released ground-truth template's largest peak-to-peak and largest
negative peak are both on the electrode at (25, -25) um, the one nearest the AIS midpoint (23.3 um
away). The next nearest, at (-25, -25), is 26.8 um away, and the second-largest peak-to-peak is 0.939
of the largest, so the margin is small on both counts.

## table1-feature-counts

> For the ground-truth model, the soma strategy uses 76 intracellular features; all adds 11
> extracellular features (87), sections adds 33 (109), and single adds 77 (153).

Data elsewhere: the repository's feature files `features_BPO_<strategy>.json` for hay_ais. The code
builds one objective per feature in these files (`define_fitness_calculator`); the criterion counts
the evaluator's objectives, which cannot be built here (LFPy simulator), so the features are counted
from the files the evaluator reads (deviation). The Methods' 79 features are shown as a variant.

In [ ]:
counts = {}
for strategy in ("soma", "all", "sections", "single"):
    f = gm.features_json(name=f"features_BPO_{strategy}.json")
    intra = sum(len(v) for p in f.values() for loc_, v in p.items() if loc_ != "MEA")
    extra = sum(len(v) for p in f.values() for loc_, v in p.items() if loc_ == "MEA")
    counts[strategy] = (intra, extra, intra + extra)
    print(f"{strategy}: intracellular {intra}, extracellular {extra}, total {intra + extra}")
expected = {"soma": 76, "all": 87, "sections": 109, "single": 153}
print("totals match Table 1:", {k: counts[k][2] == v for k, v in expected.items()})
allf = gm.features_json(name="features.json")
train = ["IDrest_150", "IDrest_250", "IDrest_300", "IV_-100", "IV_-20", "APWaveform_290"]
n79 = sum(len(v) for p in train for v in allf[p].values())
print(f"variant: features of the six training protocols in features.json before the code drops the three sag features of IV_-20: {n79}")

**Result: reproduced.** The feature files give 76 intracellular features for soma, and 87, 109 and
153 in total for all, sections and single (11, 33 and 77 extracellular). The Methods' 79 is the count
before the code drops the three sag features of IV -20%.

## The model fits (Figures 4, 5, 7, 8 and 10)

All remaining claims need the optimizations: 10 seeds per strategy, 600 generations of 20 offspring
each (`scripts/run_optimizations.py`), for four strategies on the ground-truth model and two
strategies on each experimental cell, with extracellular signals simulated for the extracellular
strategies. No optimization results are in the repository. The cost is estimated from one
evaluation of the six training protocols on the released ground-truth model, intracellular only
(the extracellular simulation adds to it), so the estimate is a lower bound.

In [ ]:
t0 = time.time()
prot = gm.protocol_json()
per = {}
for name in train:
    st = prot[name]["stimuli"][1]
    t1 = time.time()
    gm.run_step(st["amp"], st["delay"], st["duration"], st["totduration"])
    per[name] = time.time() - t1
t_eval = time.time() - t0
print("seconds per training protocol:", {k: round(v, 2) for k, v in per.items()}, f"; one evaluation {t_eval:.1f} s")
evals_gt = 10 * 4 * 600 * 20
evals_exp = 10 * 2 * 2 * 600 * 20
for label, n in (("ground truth, 4 strategies", evals_gt), ("experimental, 2 cells x 2 strategies", evals_exp)):
    cpu_h = n * t_eval / 3600
    print(f"{label}: {n} evaluations, at least {cpu_h:.0f} CPU hours, {cpu_h / 4:.0f} hours on the 4 cores of this runner")
print(f"Notebook limit: 0.5 hours; whole author run: 5 hours. ({time.time() - started:.0f} s since the start of the notebook)")

**Result: the fits are blocked as too large.** One evaluation of the six training protocols takes
about 8 s without the extracellular part, so the ground-truth fits need more than 1000 CPU hours (more
than 250 hours on this runner's 4 cores) and the experimental fits as much again, far beyond the
30-minute notebook limit and the run's 5 hours. Every claim on the fits, and those that use their solutions (Figures 7B, 7C/8C and
10), is blocked.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| recordings-two-cells-runs | dandiset | reproduced |
| ecode-protocol-structure | dandiset | reproduced |
| holding-potential | dandiset | reproduced |
| extracellular-spikes-above-50uv | dandiset | reproduced |
| mea-electrodes-and-rates | dandiset | partially reproduced |
| culture-age | dandiset | blocked: metadata insufficient |
| fig6-ais-from-dendrite | elsewhere | reproduced |
| gt-rheobase | elsewhere | reproduced |
| fig3b-ap-initiates-distal-ais | elsewhere | reproduced |
| fig3a-largest-channel-at-ais | elsewhere | reproduced |
| table1-feature-counts | elsewhere | reproduced |
| fig4a, gt-intracellular-scores, fig4c, fig4b (two), fig4d, fig5 | elsewhere | blocked: too large |
| fig78a, experimental-intracellular-scores, fig78b, fig78c, fig10 | with inputs | blocked: too large |

How the analysis differs from the paper's. The template follows Methods 5.7.1 with the code's
parameters, on the filtered MEA series of the dandiset, after the code's removal of blank channels;
the filter is applied forward and backward. The holding potential is read before the stimulus onset
found in the stimulus, 50 ms before the code's `ton`. The ground-truth model is simulated with
BluePyOpt's NEURON simulator in place of the LFPy simulator of BluePyOpt 1.13.88; the ground-truth
template is the repository's file. Table 1 is counted from the feature files, not the evaluator.

What was found about the dataset. The patch voltages are in mV and the currents in pA, though
the files say volts and amperes. Most protocols are sampled at 20.41 kHz and APWaveform at 125 kHz,
where the paper says 20 kHz and about 200 kHz. The icephys grouping tables leave out the first sweep
of most protocols and every PosCheops sweep. Many MEA channels are constant for part of a run (117 to
132 per run). In the files, the stimulus starts 50 ms before the code's `ton` timings. No culture age is given. The dandiset's metadata cite the paper as
doi:10.1162/neco_a_01676, not 10.1162/neco_a_01672.